# B Cell Merge + Visualization + scHPL treeArches v3.0

**Purpose**: Run the **global lineage-wise** scHPL/treeArches workflow on B-cell reference/query data and compare base-model labels with hierarchical scHPL validation  
**Mode**: one lineage → one tree → one query prediction pass  
**Version**: 3.0 (v2.0 base + treeArches scHPL integration)  
**Date**: 2026-03-30  
**Author**: r2end

---
## Unified methodology

- This notebook follows the shared `schpl` convention in this folder: scHPL is a **secondary hierarchical validator** after the base model, not a replacement for scANVI/scArches.
- Training and prediction are performed in the shared integrated latent (`X_scANVI_L2` in this notebook); UMAP is used only for visualization.
- Recommended scHPL setup kept here: **kNN** classifier, `dimred=True`, `useRE=True`, `n_neighbors=50`, `FN=0.5`, `rej_threshold=0.5`.
- Standard output columns written to query cells are `schpl_pred_raw`, `schpl_pred`, `schpl_prob`, `schpl_rejected`, `schpl_reject_type`, `leiden_schpl_qc`, and `schpl_novel_candidate`.
- `Rejected` means the cell is **not stably absorbed by the current B-cell reference hierarchy**; it is a follow-up flag, not an automatic new-cell-type claim.

## Changes over v2.0

**New (v3.0):**
1. scHPL hierarchical classifier trained on reference `X_scANVI_L2` latent
2. Query cells predicted with three-criterion rejection (dist / RE / prob)
3. Novel cell type candidate detection via Leiden cluster enrichment
4. `schpl_*` columns appended to query obs BEFORE merge → carried into merged object
5. scHPL visualization panels added (Section 19–21)

**Preserved from v2.0:**
- All P0/P1 fixes unchanged
- All existing visualization sections unchanged
- All obs column names from scArches mapping unchanged

---

## Section 1: Imports and Setup

In [30]:
import sys
import os
import re
from pathlib import Path
import warnings
import json
import gc
import time
import pickle
from datetime import datetime

import numpy as np
import pandas as pd
from pandas.api.types import CategoricalDtype
from scipy.sparse import issparse, csr_matrix

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

import scanpy as sc

# scHPL modules
from scHPL import train as schpl_train
from scHPL import predict as schpl_predict
from scHPL import utils as schpl_utils

warnings.filterwarnings('ignore')

PIPELINE_START = time.time()

print("=" * 80)
print("B Cell Merge + Visualization + scHPL treeArches v3.0")
print("=" * 80)
print(f"scanpy : {sc.__version__}")
print(f"pandas : {pd.__version__}")
print(f"numpy  : {np.__version__}")
try:
    import scHPL
    print(f"scHPL  : {scHPL.__version__}")
except AttributeError:
    print("scHPL  : imported (version attr not available)")
print(f"Python : {sys.version}")

B Cell Merge + Visualization + scHPL treeArches v3.0
scanpy : 1.11.5
pandas : 1.5.3
numpy  : 1.26.4
scHPL  : imported (version attr not available)
Python : 3.10.20 | packaged by conda-forge | (main, Mar  5 2026, 16:42:22) [GCC 14.3.0]


## Section 2: Configuration

In [31]:
# ===== PATH CONFIGURATION =====
PATH_REF = "/home/h2048/data/py/0203/bcell_scarches_v4_1/models/scanvi_bcell_L2_v2_5_3/reference_with_L2_umap.h5ad"
PATH_QRY = "/home/h2048/data/py/0204/scarches_mapping_L2_v2_5_3/query_mapped_L2.h5ad"
PATH_OUT = "/home/h2048/data/py/0330/bcell_merge_schpl_v3_0"

# ===== OBSKEY CONFIGURATION (from v2.0, unchanged) =====
OBSKEY_REF_LABEL  = "Cell_Type_L2"
OBSKEY_QRY_PRED   = "Cell_Type_L2_pred"
OBSKEY_QRY_FINAL  = "Cell_Type_L2_final"
OBSKEY_CONF       = "mapping_confidence"
OBSKEY_SOURCE     = "data_source"
OBSKEY_BATCH      = "sample"
OBSKEY_TISSUE     = "tissue"

# ===== VISUALIZATION COLUMN PREFIX (from v2.0, unchanged) =====
VIZ_PREFIX    = "viz"
VIZ_REF_ONLY  = f"{VIZ_PREFIX}__ref_label_only"
VIZ_QRY_FINAL = f"{VIZ_PREFIX}__qry_label_final_only"
VIZ_QRY_PRED  = f"{VIZ_PREFIX}__qry_label_pred_only"
VIZ_QRY_CONF  = f"{VIZ_PREFIX}__qry_conf_only"

# ===== MARKER GENES (from v2.0, unchanged) =====
MARKER_GENES = [
    "CD19", "MS4A1", "CD27", "IGHD", "IGHM", "MZB1", "SDC1", "JCHAIN",
]
MARKER_OBSM_KEY         = "marker_expr"
MARKER_LAYER_PREFERRED  = "log1p"
QRY_SHARED_UMAP_KEY     = "X_umap_mapped"

# ===== scHPL CONFIGURATION (new in v3.0) =====
# Latent keys: X_scANVI_L2 exists in both reference and query obsm
REF_LATENT_KEY     = "X_scANVI_L2"   # check list(adata_ref.obsm.keys()) if unsure
QRY_LATENT_KEY     = "X_scANVI_L2"   # check list(adata_qry.obsm.keys()) if unsure
UNLABELED          = "Unknown"

SCHPL_CLASSIFIER   = "knn"   # knn recommended for latent space
SCHPL_DIMRED       = True    # scHPL applies PCA internally before kNN classification
SCHPL_USE_RE       = True    # reconstruction error as 3rd rejection criterion
SCHPL_N_NEIGHBORS  = 50      # default; dynamic_neighbors=True auto-reduces for small classes
SCHPL_FN           = 0.5     # false-negative rate for RE threshold
SCHPL_REJ_THRESHOLD = 0.5   # posterior probability below this -> rejected (kNN)

# New obs columns written to query (all prefixed schpl_; v2.0 cols untouched)
COL_SCHPL_RAW      = "schpl_pred_raw"     # raw scHPL output string (accepted labels mapped back to atlas labels)
COL_SCHPL_PRED     = "schpl_pred"         # cleaned: all rejections -> 'Rejected'
COL_SCHPL_PROB     = "schpl_prob"         # max posterior probability
COL_SCHPL_REJECTED = "schpl_rejected"     # bool
COL_SCHPL_REJ_TYPE = "schpl_reject_type" # 'dist' | 'RE' | 'prob' | 'accepted'
# (leiden_schpl_qc and schpl_novel_candidate also added to query before merge)

# ===== VISUALIZATION SETTINGS =====
DPI           = 300
FIGURE_FORMAT = "pdf"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"]  = 42
sc.settings.set_figure_params(dpi=DPI, facecolor='white', format=FIGURE_FORMAT,
                               vector_friendly=False)
PALETTE_SOURCE    = {"reference": "#1f77b4", "query": "#ff7f0e"}
CMAP_CONFIDENCE   = "viridis"
CMAP_EXPRESSION   = "Reds"

# ===== REPRODUCIBILITY =====
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ===== OUTPUT DIRECTORIES =====
output_dir  = Path(PATH_OUT)
figures_dir = output_dir / "figures"
for _d in [output_dir, figures_dir]:
    _d.mkdir(parents=True, exist_ok=True)

print(f"Reference    : {PATH_REF}")
print(f"Query        : {PATH_QRY}")
print(f"Output       : {PATH_OUT}")
print(f"Ref latent   : {REF_LATENT_KEY}")
print(f"Qry latent   : {QRY_LATENT_KEY}")
print(f"Qry UMAP pref: {QRY_SHARED_UMAP_KEY}")
print(f"Marker layer : {MARKER_LAYER_PREFERRED}")
print(f"scHPL        : {SCHPL_CLASSIFIER}  dimred={SCHPL_DIMRED}  useRE={SCHPL_USE_RE}")

Reference    : /home/h2048/data/py/0203/bcell_scarches_v4_1/models/scanvi_bcell_L2_v2_5_3/reference_with_L2_umap.h5ad
Query        : /home/h2048/data/py/0204/scarches_mapping_L2_v2_5_3/query_mapped_L2.h5ad
Output       : /home/h2048/data/py/0330/bcell_merge_schpl_v3_0
Ref latent   : X_scANVI_L2
Qry latent   : X_scANVI_L2
Qry UMAP pref: X_umap_mapped
Marker layer : log1p
scHPL        : knn  dimred=True  useRE=True


## Section 3: Helper Functions (v2.0, unchanged)

In [32]:
def build_global_categories(adata, ref_label_key, qry_label_key, source_key="data_source"):
    m_ref = adata.obs[source_key].eq("reference")
    m_qry = adata.obs[source_key].eq("query")
    ref   = adata.obs.loc[m_ref, ref_label_key].astype("string").dropna()
    qry   = adata.obs.loc[m_qry, qry_label_key].astype("string").dropna()
    ref_order = pd.unique(ref)
    qry_extra = [x for x in pd.unique(qry) if x not in set(ref_order)]
    return list(ref_order) + sorted(qry_extra)


def make_palette(categories):
    n = len(categories)
    if n <= 20:
        colors = list(plt.get_cmap("tab20").colors)[:n]
    elif n <= 102:
        colors = sc.pl.palettes.default_102[:n]
    else:
        colors = [plt.cm.hsv(i / n) for i in range(n)]
        print(f"Warning: {n} categories (>102), using HSV colormap")
    return dict(zip(categories, colors))


def attach_marker_layer(adata, genes, layer_out="marker_expr", prefer_layer="log1p", allow_fallback=True):
    genes_avail = [g for g in genes if g in adata.var_names]
    if not genes_avail:
        raise ValueError("No marker genes found in var_names")

    if prefer_layer in (adata.layers or {}):
        X      = adata[:, genes_avail].layers[prefer_layer]
        source = f"layer:{prefer_layer}"
    elif allow_fallback and adata.raw is not None and all(g in adata.raw.var_names for g in genes_avail):
        X      = adata.raw[:, genes_avail].X
        source = "raw"
    elif allow_fallback:
        X      = adata[:, genes_avail].X
        source = "X"
    else:
        raise ValueError(
            f"Required comparable layer '{prefer_layer}' not found; available layers: {list(adata.layers.keys())}"
        )

    adata.obsm[layer_out]               = X.toarray() if issparse(X) else np.asarray(X)
    adata.uns[f"{layer_out}_genes"]     = genes_avail
    adata.uns[f"{layer_out}_source"]    = source
    print(f"  Stored {len(genes_avail)} markers in .obsm['{layer_out}'] (from {source})")
    return adata, genes_avail


def marker_sources_are_comparable(ref_source, qry_source):
    return (
        ref_source is not None and
        qry_source is not None and
        ref_source == qry_source and
        str(ref_source).startswith("layer:")
    )


def make_safe_newick_label_map(labels):
    labels = [str(x) for x in pd.unique(pd.Series(labels, dtype="string").dropna())]
    label_to_token = {}
    token_to_label = {}
    used = {"root", "root2"}

    for i, label in enumerate(labels):
        base = re.sub(r"[^0-9A-Za-z_]", "_", label).strip("_") or f"label_{i}"
        token = base
        suffix = 1
        while token in used:
            token = f"{base}_{suffix}"
            suffix += 1
        used.add(token)
        label_to_token[label] = token
        token_to_label[token] = label

    return label_to_token, token_to_label


def parse_schpl_predictions(y_pred_raw, known_labels, token_to_label=None):
    token_to_label = token_to_label or {}
    known_labels = {str(x) for x in known_labels}

    raw_tokens = np.asarray(y_pred_raw, dtype=str)
    raw_norm   = np.array([str(x).strip() for x in raw_tokens], dtype=object)
    raw_lower  = np.array([x.lower() for x in raw_norm], dtype=object)
    raw_mapped = np.array([token_to_label.get(x, x) for x in raw_norm], dtype=object)

    mask_rej_dist = np.array(["dist" in x for x in raw_lower], dtype=bool)
    mask_rej_re   = np.array([("re)" in x) or ("reconstruction" in x) for x in raw_lower], dtype=bool)
    mask_rej_prob = np.array([
        (x in {"root", "root2"}) or
        ("prob" in x) or
        ("reject" in x and "dist" not in x and "re)" not in x)
        for x in raw_lower
    ], dtype=bool)

    mask_rejected = mask_rej_dist | mask_rej_re | mask_rej_prob
    mask_accepted = np.array([x in known_labels for x in raw_mapped], dtype=bool) & ~mask_rejected
    mask_unknown  = ~(mask_rejected | mask_accepted)

    if mask_unknown.any():
        unknown_vals = sorted(pd.unique(raw_norm[mask_unknown]).tolist())
        raise AssertionError(
            f"Unparsed scHPL outputs encountered: {unknown_vals[:10]}"
        )

    pred_clean = raw_mapped.astype(object).copy()
    pred_clean[mask_rejected] = "Rejected"
    rej_type = np.where(mask_rej_dist, "dist",
               np.where(mask_rej_re,   "RE",
               np.where(mask_rej_prob, "prob", "accepted")))

    return {
        "raw_tokens": raw_tokens,
        "raw_mapped": raw_mapped,
        "pred_clean": pred_clean,
        "mask_rej_dist": mask_rej_dist,
        "mask_rej_re": mask_rej_re,
        "mask_rej_prob": mask_rej_prob,
        "mask_rejected": mask_rejected,
        "mask_accepted": mask_accepted,
        "rej_type": rej_type,
    }


def remove_unused_categories(adata, col):
    if col in adata.obs.columns:
        s = adata.obs[col]
        if isinstance(s.dtype, CategoricalDtype):
            adata.obs[col] = s.cat.remove_unused_categories()


def save_rasterized_figure(fig, path, dpi=300, rasterize_scatter=True):
    if rasterize_scatter:
        for ax in fig.axes:
            for coll in ax.collections:
                coll.set_rasterized(True)
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    plt.close(fig)


print("Helper functions loaded (v2.0 unchanged + v3.0 scHPL helpers below)")

Helper functions loaded (v2.0 unchanged + v3.0 scHPL helpers below)


## Section 4: Load Reference Data

In [33]:
print("=" * 80)
print("STEP 1: Load Reference Data")
print("=" * 80)

path_ref = Path(PATH_REF)
if not path_ref.exists():
    raise FileNotFoundError(f"[ERROR] Reference h5ad not found: {path_ref}")

adata_ref = sc.read_h5ad(path_ref)
adata_ref.var_names_make_unique()
adata_ref.obs_names_make_unique()

print(f"Reference shape : {adata_ref.shape}")
print(f"obsm keys       : {list(adata_ref.obsm.keys())}")

if OBSKEY_REF_LABEL not in adata_ref.obs.columns:
    raise ValueError(f"Reference missing '{OBSKEY_REF_LABEL}'")
if "X_umap" not in adata_ref.obsm:
    raise ValueError("Reference missing 'X_umap'")
if REF_LATENT_KEY not in adata_ref.obsm:
    raise KeyError(
        f"REF_LATENT_KEY='{REF_LATENT_KEY}' not in obsm.\n"
        f"Available: {list(adata_ref.obsm.keys())}\n"
        f"Update REF_LATENT_KEY in Section 2."
    )

print(f"\nLabel distribution (top 8):")
for ct, n in adata_ref.obs[OBSKEY_REF_LABEL].value_counts().head(8).items():
    print(f"  {ct}: {n:,} ({n/adata_ref.n_obs*100:.1f}%)")

STEP 1: Load Reference Data


Reference shape : (11570, 4020)
obsm keys       : ['X_cnmf_usages', 'X_harmony', 'X_pca', 'X_scANVI_L2', 'X_scanvi', 'X_scanvi_corrected', 'X_scvi', 'X_umap', 'X_umap_scanvi', 'X_umap_scanvi_corrected', 'X_umap_scvi', '_scvi_extra_categorical_covs', '_scvi_extra_continuous_covs']

Label distribution (top 8):
  GC_B: 3,547 (30.7%)
  Naive_B: 3,012 (26.0%)
  Memory_B: 2,509 (21.7%)
  Plasma: 2,048 (17.7%)
  Atypical_Memory_B: 447 (3.9%)
  Unknown: 7 (0.1%)


## Section 5: Attach Marker Expression to Reference

In [34]:
print("=" * 80)
print("STEP 2: Attach Marker Expression (Reference)")
print("=" * 80)

adata_ref, markers_ref = attach_marker_layer(
    adata_ref,
    MARKER_GENES,
    layer_out=MARKER_OBSM_KEY,
    prefer_layer=MARKER_LAYER_PREFERRED,
    allow_fallback=False,
)
MARKER_SOURCE_REF = adata_ref.uns.get(f"{MARKER_OBSM_KEY}_source")
print(f"Reference: {len(markers_ref)} markers stored in .obsm['{MARKER_OBSM_KEY}']")
print(f"Reference marker source: {MARKER_SOURCE_REF}")

STEP 2: Attach Marker Expression (Reference)
  Stored 8 markers in .obsm['marker_expr'] (from layer:log1p)
Reference: 8 markers stored in .obsm['marker_expr']
Reference marker source: layer:log1p


## Section 6: Load Query Data

In [35]:
print("=" * 80)
print("STEP 3: Load Query Data")
print("=" * 80)

path_qry = Path(PATH_QRY)
if not path_qry.exists():
    raise FileNotFoundError(f"[ERROR] Query h5ad not found: {path_qry}")

adata_query = sc.read_h5ad(path_qry)
adata_query.var_names_make_unique()
adata_query.obs_names_make_unique()

print(f"Query shape  : {adata_query.shape}")
print(f"obsm keys    : {list(adata_query.obsm.keys())}")

required_qry = [OBSKEY_QRY_PRED, OBSKEY_QRY_FINAL, OBSKEY_CONF]
missing = [k for k in required_qry if k not in adata_query.obs.columns]
if missing:
    raise ValueError(f"Query missing columns: {missing}")
if "X_umap" not in adata_query.obsm and QRY_SHARED_UMAP_KEY not in adata_query.obsm:
    raise ValueError(
        f"Query missing both 'X_umap' and '{QRY_SHARED_UMAP_KEY}', cannot build shared merged UMAP"
    )
if QRY_LATENT_KEY not in adata_query.obsm:
    raise KeyError(
        f"QRY_LATENT_KEY='{QRY_LATENT_KEY}' not in query obsm.\n"
        f"Available: {list(adata_query.obsm.keys())}\n"
        f"Update QRY_LATENT_KEY in Section 2."
    )

if QRY_SHARED_UMAP_KEY in adata_query.obsm:
    adata_query.obsm["X_umap"] = np.asarray(adata_query.obsm[QRY_SHARED_UMAP_KEY]).copy()
    print(f"  [OK] Using '{QRY_SHARED_UMAP_KEY}' as shared ref/query UMAP coordinates")
else:
    print(f"  [WARN] '{QRY_SHARED_UMAP_KEY}' not found; merged ref/query UMAP may be incomparable.")

new_cols = [
    COL_SCHPL_RAW,
    COL_SCHPL_PRED,
    COL_SCHPL_PROB,
    COL_SCHPL_REJECTED,
    COL_SCHPL_REJ_TYPE,
    "leiden_schpl_qc",
    "schpl_novel_candidate",
]
collisions = [k for k in new_cols if k in adata_query.obs.columns]
if collisions:
    print(f"  [INFO] Overwriting existing scHPL columns: {collisions}")

print(f"\nQuery label distribution (top 8):")
conf_vals = pd.to_numeric(adata_query.obs[OBSKEY_CONF], errors='coerce')
for ct, n in adata_query.obs[OBSKEY_QRY_FINAL].value_counts().head(8).items():
    print(f"  {ct}: {n:,} ({n/adata_query.n_obs*100:.1f}%)")
print(f"\nMapping confidence: mean={conf_vals.mean():.3f}  "
      f"median={conf_vals.median():.3f}  "
      f"low(<0.5)={(conf_vals<0.5).sum():,}")

STEP 3: Load Query Data
Query shape  : (41217, 83690)
obsm keys    : ['X_scANVI_L2', 'X_scANVI_mapped', 'X_umap', 'X_umap_mapped', '_scvi_extra_categorical_covs', '_scvi_extra_continuous_covs']
  [OK] Using 'X_umap_mapped' as shared ref/query UMAP coordinates

Query label distribution (top 8):
  Plasma: 19,902 (48.3%)
  Naive_B: 11,272 (27.3%)
  Memory_B: 9,126 (22.1%)
  Atypical_Memory_B: 630 (1.5%)
  GC_B: 155 (0.4%)
  Unknown: 132 (0.3%)

Mapping confidence: mean=0.971  median=1.000  low(<0.5)=132


## Section 7: Attach Marker Expression to Query

In [36]:
print("=" * 80)
print("STEP 4: Attach Marker Expression (Query)")
print("=" * 80)

adata_query, markers_qry = attach_marker_layer(
    adata_query,
    MARKER_GENES,
    layer_out=MARKER_OBSM_KEY,
    prefer_layer=MARKER_LAYER_PREFERRED,
    allow_fallback=True,
)
MARKER_SOURCE_QRY = adata_query.uns.get(f"{MARKER_OBSM_KEY}_source")
MARKER_EXPR_COMPARABLE = marker_sources_are_comparable(MARKER_SOURCE_REF, MARKER_SOURCE_QRY)

print(f"Query: {len(markers_qry)} markers stored in .obsm['{MARKER_OBSM_KEY}']")
print(f"Query marker source: {MARKER_SOURCE_QRY}")

markers_common = sorted(set(markers_ref) & set(markers_qry))
print(f"Common markers for merge: {len(markers_common)}  {markers_common}")

if MARKER_EXPR_COMPARABLE:
    print(f"  [OK] Marker sources match ({MARKER_SOURCE_REF}); merged marker panels enabled")
else:
    print(
        f"  [WARN] Marker sources differ (ref={MARKER_SOURCE_REF}, query={MARKER_SOURCE_QRY}); "
        f"merged marker panels will be disabled to avoid cross-source color misinterpretation"
    )

STEP 4: Attach Marker Expression (Query)
  Stored 8 markers in .obsm['marker_expr'] (from X)
Query: 8 markers stored in .obsm['marker_expr']
Query marker source: X
Common markers for merge: 8  ['CD19', 'CD27', 'IGHD', 'IGHM', 'JCHAIN', 'MS4A1', 'MZB1', 'SDC1']
  [WARN] Marker sources differ (ref=layer:log1p, query=X); merged marker panels will be disabled to avoid cross-source color misinterpretation


## Section 8: scHPL — Train Hierarchical Classifier on Reference

Runs on `X_scANVI_L2` latent (already computed; no model retraining).  
Trains a kNN-based classification tree on reference L2 labels.

In [37]:
print("=" * 80)
print("STEP 5: scHPL TRAIN (reference latent)")
print("=" * 80)

# Extract reference latent + labels
X_ref = adata_ref.obsm[REF_LATENT_KEY].astype(np.float32)
y_ref_original = adata_ref.obs[OBSKEY_REF_LABEL].astype(str).values

# Drop Unknown from reference training set
known_mask    = y_ref_original != UNLABELED
n_unknown_ref = (~known_mask).sum()
if n_unknown_ref > 0:
    print(f"  Dropping {n_unknown_ref} Unknown cells from reference training")
X_ref = X_ref[known_mask]
y_ref_original = y_ref_original[known_mask]

print(f"  Reference latent shape : {X_ref.shape}")
print(f"  Unique L2 classes      : {len(np.unique(y_ref_original))}")

# Warn about small classes (dynamic_neighbors handles them automatically)
class_counts  = pd.Series(y_ref_original).value_counts()
small_classes = class_counts[class_counts < SCHPL_N_NEIGHBORS]
if len(small_classes) > 0:
    print(f"  [{len(small_classes)} classes < {SCHPL_N_NEIGHBORS} cells; "
          f"dynamic_neighbors=True will adjust]")
    for lbl, n in small_classes.items():
        print(f"    {lbl}: {n}")

# Build initial flat tree and train
# scHPL 1.0.x expects Newick-safe labels for utils.create_tree(); keep reverse map for reporting.
label_to_token, token_to_label = make_safe_newick_label_map(y_ref_original)
y_ref = np.array([label_to_token[x] for x in y_ref_original], dtype=object)
unique_tokens = np.array([label_to_token[x] for x in np.unique(y_ref_original)], dtype=object)

print(f"\n  Building initial flat tree from reference labels...")
tree_newick = f"({','.join(unique_tokens)})root;"
tree_init   = schpl_utils.create_tree(tree_newick)
print(f"  Initial tree           : {tree_init}")
if any(label_to_token[k] != k for k in label_to_token):
    print("  [INFO] Applied Newick-safe label tokenization for scHPL tree construction")

print(f"\n  Training scHPL tree...")
print(f"  classifier={SCHPL_CLASSIFIER}  dimred={SCHPL_DIMRED}  "
      f"useRE={SCHPL_USE_RE}  n_neighbors={SCHPL_N_NEIGHBORS}")

t0           = time.time()
tree_trained = schpl_train.train_tree(
    data              = X_ref,
    labels            = y_ref,
    tree              = tree_init,
    classifier        = SCHPL_CLASSIFIER,
    dimred            = SCHPL_DIMRED,
    useRE             = SCHPL_USE_RE,
    FN                = SCHPL_FN,
    n_neighbors       = SCHPL_N_NEIGHBORS,
    dynamic_neighbors = True,
)
print(f"  [OK] Training done: {time.time()-t0:.1f} s")
print(f"  Trained tree         : {tree_trained}")

# Save tree immediately
tree_path = output_dir / "schpl_tree_bcell_L2_v3_0.pkl"
with open(tree_path, "wb") as f:
    pickle.dump(tree_trained, f)
print(f"  [OK] Tree saved: {tree_path}")

STEP 5: scHPL TRAIN (reference latent)
  Dropping 7 Unknown cells from reference training
  Reference latent shape : (11563, 50)
  Unique L2 classes      : 5

  Building initial flat tree from reference labels...
  Initial tree           : [Node("['root']")]

  Training scHPL tree...
  classifier=knn  dimred=True  useRE=True  n_neighbors=50
  [OK] Training done: 6.7 s
  Trained tree         : [Node("['root']")]
  [OK] Tree saved: /home/h2048/data/py/0330/bcell_merge_schpl_v3_0/schpl_tree_bcell_L2_v3_0.pkl


## Section 9: scHPL — Predict on Query + Write obs Columns

In [38]:
print("=" * 80)
print("STEP 6: scHPL PREDICT (query) + WRITE OBS COLUMNS")
print("=" * 80)

X_qry = adata_query.obsm[QRY_LATENT_KEY].astype(np.float32)
print(f"  Query latent shape : {X_qry.shape}")

t0 = time.time()
y_pred_raw, y_prob_raw = schpl_predict.predict_labels(
    X_qry,
    tree=tree_trained,
    threshold=SCHPL_REJ_THRESHOLD,
)
elapsed_pred = time.time() - t0
print(f"  [OK] Prediction done: {elapsed_pred:.1f} s")
print(f"  y_pred_raw shape : {np.asarray(y_pred_raw).shape}")
print(f"  y_prob_raw shape : {np.asarray(y_prob_raw).shape if y_prob_raw is not None else 'None'}")

# ---- Process rejection types ----
parsed = parse_schpl_predictions(
    y_pred_raw,
    known_labels=np.unique(y_ref_original),
    token_to_label=token_to_label,
)
y_pred_arr    = np.asarray(parsed["raw_mapped"], dtype=object)
y_pred_clean  = np.asarray(parsed["pred_clean"], dtype=object)
mask_rej_dist = parsed["mask_rej_dist"]
mask_rej_re   = parsed["mask_rej_re"]
mask_rej_prob = parsed["mask_rej_prob"]
mask_rejected = parsed["mask_rejected"]
mask_accepted = parsed["mask_accepted"]
rej_type      = parsed["rej_type"]

# ---- Write to query obs ----
adata_query.obs[COL_SCHPL_RAW]      = y_pred_arr
adata_query.obs[COL_SCHPL_PRED]     = y_pred_clean
adata_query.obs[COL_SCHPL_REJECTED] = mask_rejected

if y_prob_raw is not None:
    y_prob_arr = np.asarray(y_prob_raw, dtype=np.float32)
    if y_prob_arr.ndim == 2:
        y_prob_arr = y_prob_arr[:, 0]
    adata_query.obs[COL_SCHPL_PROB] = y_prob_arr
else:
    adata_query.obs[COL_SCHPL_PROB] = np.nan

adata_query.obs[COL_SCHPL_REJ_TYPE] = rej_type

n_total    = adata_query.n_obs
n_rejected = int(mask_rejected.sum())
n_accepted = int(mask_accepted.sum())
rej_rate   = n_rejected / n_total * 100

print(f"\n  Total query cells : {n_total:,}")
print(f"  Accepted          : {n_accepted:,} ({n_accepted/n_total*100:.1f}%)")
print(f"  Rejected (all)    : {n_rejected:,} ({rej_rate:.1f}%)")
print(f"    dist            : {int(mask_rej_dist.sum()):,}")
print(f"    RE              : {int(mask_rej_re.sum()):,}")
print(f"    prob            : {int(mask_rej_prob.sum()):,}")

if rej_rate > 30:
    print(f"  [WARN] >30% rejection. May indicate novel populations or latent key mismatch.")
elif rej_rate < 1:
    print(f"  [INFO] <1% rejection. Query closely matches reference.")
else:
    print(f"  [OK] Rejection rate {rej_rate:.1f}% within expected range.")

print(f"\n  schpl_pred distribution:")
for lbl, n in pd.Series(y_pred_clean).value_counts().items():
    print(f"    {lbl}: {n:,}")

STEP 6: scHPL PREDICT (query) + WRITE OBS COLUMNS
  Query latent shape : (41217, 50)


  [OK] Prediction done: 205.2 s
  y_pred_raw shape : (41217,)
  y_prob_raw shape : (41217, 1)

  Total query cells : 41,217
  Accepted          : 40,181 (97.5%)
  Rejected (all)    : 1,036 (2.5%)
    dist            : 174
    RE              : 72
    prob            : 790
  [OK] Rejection rate 2.5% within expected range.

  schpl_pred distribution:
    Plasma: 19,706
    Memory_B: 11,464
    Naive_B: 8,212
    Rejected: 1,036
    Atypical_Memory_B: 630
    GC_B: 169


## Section 10: scHPL — Novel Cell Type Candidate Detection

In [39]:
print("=" * 80)
print("STEP 7: NOVEL CELL TYPE CANDIDATE DETECTION")
print("=" * 80)

# Use existing X_umap from query; build neighbors on X_scANVI_L2
if "neighbors_scanvi_L2" not in adata_query.uns:
    sc.pp.neighbors(adata_query, use_rep=QRY_LATENT_KEY, n_neighbors=30,
                    random_state=RANDOM_SEED, key_added="neighbors_scanvi_L2")

sc.tl.leiden(adata_query, resolution=0.5, key_added="leiden_schpl_qc",
             neighbors_key="neighbors_scanvi_L2", random_state=RANDOM_SEED)
print(f"  Leiden clusters (res=0.5): {adata_query.obs['leiden_schpl_qc'].nunique()}")

# Rejection rate per Leiden cluster
cluster_rej = (
    adata_query.obs.groupby("leiden_schpl_qc")[COL_SCHPL_REJECTED]
    .agg(n_rejected="sum", n_total="count", rej_rate="mean")
    .sort_values("rej_rate", ascending=False)
)
cluster_rej["rej_rate_pct"] = (cluster_rej["rej_rate"] * 100).round(1)

# Novel candidate: cluster rej_rate > 50% AND >= 50 rejected cells
NOVEL_REJ_THRESHOLD = 0.50
NOVEL_MIN_CELLS     = 50
novel_clusters = cluster_rej[
    (cluster_rej["rej_rate"] > NOVEL_REJ_THRESHOLD) &
    (cluster_rej["n_rejected"] >= NOVEL_MIN_CELLS)
].index.tolist()

adata_query.obs["schpl_novel_candidate"] = (
    adata_query.obs["leiden_schpl_qc"].isin(novel_clusters) &
    adata_query.obs[COL_SCHPL_REJECTED]
)

print(f"  Novel candidate clusters (rej>{NOVEL_REJ_THRESHOLD*100:.0f}%, n>={NOVEL_MIN_CELLS}):")
if not novel_clusters:
    print("    None. Rejected cells are scattered -> likely technical noise.")
else:
    for c in novel_clusters:
        r = cluster_rej.loc[c]
        print(f"    Cluster {c}: {int(r['n_total']):,} cells, "
              f"{int(r['n_rejected']):,} rejected ({r['rej_rate_pct']:.1f}%)")
    print(f"  Total novel candidate cells: "
          f"{adata_query.obs['schpl_novel_candidate'].sum():,}")

# Cluster rejection summary table
cluster_rej.to_csv(output_dir / "cluster_rejection_summary_v3_0.csv")
print(f"  [OK] cluster_rejection_summary_v3_0.csv")

STEP 7: NOVEL CELL TYPE CANDIDATE DETECTION
  Leiden clusters (res=0.5): 8
  Novel candidate clusters (rej>50%, n>=50):
    None. Rejected cells are scattered -> likely technical noise.
  [OK] cluster_rejection_summary_v3_0.csv


## Section 11: scHPL — Compare with scArches Predictions

In [40]:
print("=" * 80)
print("STEP 8: COMPARE scHPL vs scArches PREDICTIONS")
print("=" * 80)

df_accepted = adata_query.obs.loc[
    ~adata_query.obs[COL_SCHPL_REJECTED],
    [OBSKEY_QRY_PRED, OBSKEY_QRY_FINAL, OBSKEY_CONF, COL_SCHPL_PRED]
].copy()

agree_pred = df_accepted[OBSKEY_QRY_PRED].astype(str) == df_accepted[COL_SCHPL_PRED].astype(str)
agree_final = df_accepted[OBSKEY_QRY_FINAL].astype(str) == df_accepted[COL_SCHPL_PRED].astype(str)
agree_rate_pred = agree_pred.mean() * 100
agree_rate_final = agree_final.mean() * 100

print(f"  Accepted cells (scHPL)               : {len(df_accepted):,}")
print(f"  Agreement (scArches pred vs scHPL)   : {agree_rate_pred:.1f}%")
print(f"  Agreement (scArches final vs scHPL)  : {agree_rate_final:.1f}%")

# Top disagreement pairs against raw scArches prediction
df_disagree = df_accepted[~agree_pred]
if len(df_disagree) > 0:
    print(f"  Top disagreement pairs (scArches_pred -> scHPL_pred):")
    pairs = (
        df_disagree.groupby([OBSKEY_QRY_PRED, COL_SCHPL_PRED])
        .size().reset_index(name='n').sort_values('n', ascending=False)
    )
    for _, row in pairs.head(8).iterrows():
        print(f"    {row[OBSKEY_QRY_PRED]} -> {row[COL_SCHPL_PRED]}: {row['n']:,}")

# scArches labels for rejected cells
df_rejected = adata_query.obs[adata_query.obs[COL_SCHPL_REJECTED]]
if len(df_rejected) > 0:
    print(f"\n  scArches final label dist for scHPL-rejected cells (top 8):")
    for lbl, n in df_rejected[OBSKEY_QRY_FINAL].value_counts().head(8).items():
        print(f"    {lbl}: {n:,} ({n/len(df_rejected)*100:.1f}%)")

STEP 8: COMPARE scHPL vs scArches PREDICTIONS
  Accepted cells (scHPL)               : 40,181
  Agreement (scArches pred vs scHPL)   : 92.6%
  Agreement (scArches final vs scHPL)  : 92.4%
  Top disagreement pairs (scArches_pred -> scHPL_pred):
    Naive_B -> Memory_B: 2,463
    Atypical_Memory_B -> Memory_B: 125
    Memory_B -> Atypical_Memory_B: 95
    Naive_B -> Atypical_Memory_B: 82
    Memory_B -> Naive_B: 63
    Plasma -> Memory_B: 47
    Memory_B -> Plasma: 31
    Naive_B -> Plasma: 20

  scArches final label dist for scHPL-rejected cells (top 8):
    Naive_B: 578 (55.8%)
    Plasma: 213 (20.6%)
    Memory_B: 141 (13.6%)
    Atypical_Memory_B: 70 (6.8%)
    Unknown: 25 (2.4%)
    GC_B: 9 (0.9%)


## Section 12: Gene Alignment and Merge

scHPL columns are now in `adata_query.obs` and will be preserved through `sc.concat`.

In [41]:
print("=" * 80)
print("STEP 9: GENE ALIGNMENT + MERGE")
print("=" * 80)

ref_genes    = list(adata_ref.var_names)
common_genes = [g for g in ref_genes if g in adata_query.var_names]
overlap_pct  = len(common_genes) / len(ref_genes) * 100

print(f"  Reference genes : {len(ref_genes):,}")
print(f"  Common genes    : {len(common_genes):,}  ({overlap_pct:.1f}%)")
if overlap_pct < 95.0:
    print(f"  [WARN] Low overlap ({overlap_pct:.1f}%).")

adata_ref_m = adata_ref[:, common_genes].copy()
adata_qry_m = adata_query[:, common_genes].copy()

assert adata_ref_m.var_names.equals(adata_qry_m.var_names), "ref/qry var_names mismatch after alignment"

# Keep only essential obsm keys
keep_obsm = {"X_umap", QRY_LATENT_KEY}
if MARKER_EXPR_COMPARABLE:
    keep_obsm.add(MARKER_OBSM_KEY)
for adata in [adata_ref_m, adata_qry_m]:
    for key in list(adata.obsm.keys()):
        if key not in keep_obsm:
            del adata.obsm[key]
    adata.uns = {}
    if issparse(adata.X):
        adata.X = csr_matrix(adata.X)

# Unique obs_names to avoid collision
adata_ref_m.obs_names = [f"ref::{x}" for x in adata_ref_m.obs_names]
adata_qry_m.obs_names = [f"qry::{x}" for x in adata_qry_m.obs_names]

# Cross-fill missing columns so concat is clean
# Reference needs query-specific columns (NA for ref cells)
schpl_new_cols = [COL_SCHPL_RAW, COL_SCHPL_PRED, COL_SCHPL_REJ_TYPE,
                  "leiden_schpl_qc", "schpl_novel_candidate"]
for col in [OBSKEY_QRY_PRED, OBSKEY_QRY_FINAL] + schpl_new_cols:
    if col not in adata_ref_m.obs.columns:
        adata_ref_m.obs[col] = pd.NA
for col in [OBSKEY_CONF, COL_SCHPL_PROB]:
    if col not in adata_ref_m.obs.columns:
        adata_ref_m.obs[col] = np.nan
for col in [COL_SCHPL_REJECTED, "schpl_novel_candidate"]:
    if col not in adata_ref_m.obs.columns:
        adata_ref_m.obs[col] = False

# Query needs reference label column
if OBSKEY_REF_LABEL not in adata_qry_m.obs.columns:
    adata_qry_m.obs[OBSKEY_REF_LABEL] = pd.NA

# Merge
adata_merged = sc.concat(
    {"reference": adata_ref_m, "query": adata_qry_m},
    axis=0, join="inner", merge="unique", label=OBSKEY_SOURCE
)

print(f"\n  Merged shape : {adata_merged.shape}")
print(f"  Source dist  : {adata_merged.obs[OBSKEY_SOURCE].value_counts().to_dict()}")

# Verify obsm integrity
expected_obsm = ["X_umap"] + ([MARKER_OBSM_KEY] if MARKER_EXPR_COMPARABLE else [])
for key in expected_obsm:
    assert key in adata_merged.obsm, f"{key} lost during merge!"
    assert adata_merged.obsm[key].shape[0] == adata_merged.n_obs
    print(f"  obsm['{key}']: {adata_merged.obsm[key].shape}  OK")

if MARKER_EXPR_COMPARABLE:
    if f"{MARKER_OBSM_KEY}_genes" not in adata_merged.uns:
        adata_merged.uns[f"{MARKER_OBSM_KEY}_genes"] = markers_common
else:
    adata_merged.uns["marker_expr_disabled_reason"] = (
        f"Merged marker panels disabled because ref source={MARKER_SOURCE_REF}, query source={MARKER_SOURCE_QRY}"
    )

adata_merged.uns["query_umap_basis"] = QRY_SHARED_UMAP_KEY if QRY_SHARED_UMAP_KEY in adata_query.obsm else "X_umap"

print(f"  query UMAP basis for merged plots: {adata_merged.uns['query_umap_basis']}")
if not MARKER_EXPR_COMPARABLE:
    print(f"  [INFO] Merged marker panels disabled: ref={MARKER_SOURCE_REF}, query={MARKER_SOURCE_QRY}")

del adata_ref_m, adata_qry_m
gc.collect()
print("  [OK] Merge complete")

STEP 9: GENE ALIGNMENT + MERGE
  Reference genes : 4,020
  Common genes    : 3,955  (98.4%)

  Merged shape : (52787, 3955)
  Source dist  : {'query': 41217, 'reference': 11570}
  obsm['X_umap']: (52787, 2)  OK
  query UMAP basis for merged plots: X_umap_mapped
  [INFO] Merged marker panels disabled: ref=layer:log1p, query=X
  [OK] Merge complete


## Section 13: Build Global Categories + Fixed Palette

In [42]:
print("=" * 80)
print("STEP 10: GLOBAL CATEGORIES + PALETTE (P0-2 FIX)")
print("=" * 80)

GLOBAL_CATS = build_global_categories(
    adata_merged, OBSKEY_REF_LABEL, OBSKEY_QRY_FINAL, OBSKEY_SOURCE
)
CT_PALETTE  = make_palette(GLOBAL_CATS)

# Extended palette including explicit grey for 'Rejected'
GLOBAL_CATS_EXT = GLOBAL_CATS + (["Rejected"] if "Rejected" not in GLOBAL_CATS else [])
CT_PALETTE_EXT  = make_palette(GLOBAL_CATS_EXT)
CT_PALETTE_EXT["Rejected"] = "#BDBDBD"

print(f"  Categories       : {len(GLOBAL_CATS)}  {GLOBAL_CATS}")
print(f"  Extended (+Rej.) : {len(GLOBAL_CATS_EXT)}")
print(f"  Rejected color   : {CT_PALETTE_EXT['Rejected']}")

STEP 10: GLOBAL CATEGORIES + PALETTE (P0-2 FIX)
  Categories       : 6  ['Atypical_Memory_B', 'GC_B', 'Memory_B', 'Plasma', 'Naive_B', 'Unknown']
  Extended (+Rej.) : 7
  Rejected color   : #BDBDBD


## Section 14: Create Visualization Columns

In [43]:
print("=" * 80)
print("STEP 11: VISUALIZATION COLUMNS (P0-1/P0-2 FIX)")
print("=" * 80)

ref_mask = adata_merged.obs[OBSKEY_SOURCE].eq("reference")
qry_mask = adata_merged.obs[OBSKEY_SOURCE].eq("query")

def _make_label_col(adata, src_col, mask, cats):
    s = pd.Series(pd.NA, index=adata.obs_names, dtype="string")
    if src_col in adata.obs.columns:
        s.loc[mask] = adata.obs.loc[mask, src_col].astype("string")
    return pd.Categorical(s, categories=cats)

adata_merged.obs[VIZ_QRY_FINAL] = _make_label_col(
    adata_merged, OBSKEY_QRY_FINAL, qry_mask, GLOBAL_CATS)
adata_merged.obs[VIZ_QRY_PRED]  = _make_label_col(
    adata_merged, OBSKEY_QRY_PRED,  qry_mask, GLOBAL_CATS)
adata_merged.obs[VIZ_REF_ONLY]  = _make_label_col(
    adata_merged, OBSKEY_REF_LABEL, ref_mask, GLOBAL_CATS)

conf_arr = np.full(adata_merged.n_obs, np.nan)
if OBSKEY_CONF in adata_merged.obs.columns:
    conf_arr[qry_mask] = pd.to_numeric(
        adata_merged.obs.loc[qry_mask, OBSKEY_CONF], errors='coerce').values
adata_merged.obs[VIZ_QRY_CONF] = conf_arr

# schpl_pred visualization column (ref cells = NA, query cells = pred/Rejected)
VIZ_SCHPL_PRED = "viz__schpl_pred_qry_only"
adata_merged.obs[VIZ_SCHPL_PRED] = _make_label_col(
    adata_merged, COL_SCHPL_PRED, qry_mask, GLOBAL_CATS_EXT)

print(f"  viz columns created: {VIZ_REF_ONLY}, {VIZ_QRY_FINAL}, "
      f"{VIZ_QRY_PRED}, {VIZ_QRY_CONF}, {VIZ_SCHPL_PRED}")

STEP 11: VISUALIZATION COLUMNS (P0-1/P0-2 FIX)
  viz columns created: viz__ref_label_only, viz__qry_label_final_only, viz__qry_label_pred_only, viz__qry_conf_only, viz__schpl_pred_qry_only


## Section 15: Save Merged Data + Configuration

In [44]:
print("=" * 80)
print("STEP 12: SAVE Merged DATA + CONFIG")
print("=" * 80)

# Sanitize obs dtypes before writing h5ad
# AnnData/h5py can fail on mixed object columns such as bool + NA after concat,
# and this environment also dislikes pandas nullable string arrays.
for col in adata_merged.obs.columns:
    s = adata_merged.obs[col]
    dtype_str = str(s.dtype)
    if dtype_str.startswith("string"):
        adata_merged.obs[col] = s.fillna("").astype(str)
        print(f"  [sanitize] {col}: pandas string -> plain str")
    elif s.dtype == object:
        non_null = s.dropna()
        py_types = set(non_null.map(lambda x: type(x).__name__)) if len(non_null) > 0 else set()
        if py_types <= {"bool"}:
            adata_merged.obs[col] = s.fillna(False).astype(bool)
            print(f"  [sanitize] {col}: object(bool+NA) -> bool")
        elif py_types <= {"str"}:
            adata_merged.obs[col] = s.where(s.notna(), "").astype(str)
            print(f"  [sanitize] {col}: object(str) -> plain str")

merged_output = output_dir / "reference_plus_query_merged_v3_0.h5ad"
adata_merged.write_h5ad(merged_output, compression='gzip')
print(f"  [OK] {merged_output}  "
      f"({merged_output.stat().st_size/1024**3:.2f} GB)")

viz_config = {
    "pipeline":    "B_Cell_Merge_Visualization_scHPL",
    "version":     "3.0",
    "timestamp":   datetime.now().isoformat(),
    "input_ref":   PATH_REF,
    "input_qry":   PATH_QRY,
    "output_dir":  PATH_OUT,
    "query_umap_basis": adata_merged.uns.get("query_umap_basis", "X_umap"),
    "marker_expr": {
        "preferred_layer": MARKER_LAYER_PREFERRED,
        "reference_source": MARKER_SOURCE_REF,
        "query_source": MARKER_SOURCE_QRY,
        "comparable_for_merged_plots": bool(MARKER_EXPR_COMPARABLE),
    },
    "schpl": {
        "classifier":       SCHPL_CLASSIFIER,
        "dimred":           SCHPL_DIMRED,
        "useRE":            SCHPL_USE_RE,
        "n_neighbors":      SCHPL_N_NEIGHBORS,
        "FN":               SCHPL_FN,
        "rej_threshold":    SCHPL_REJ_THRESHOLD,
        "ref_latent_key":   REF_LATENT_KEY,
        "qry_latent_key":   QRY_LATENT_KEY,
        "n_ref_cells":      int(X_ref.shape[0]),
        "n_ref_classes":    int(len(np.unique(y_ref_original))),
        "n_rejected":       n_rejected,
        "rejection_rate":   round(rej_rate, 3),
        "novel_clusters":   novel_clusters,
        "agreement_pred":   round(float(agree_rate_pred), 3),
        "agreement_final":  round(float(agree_rate_final), 3),
    },
    "global_categories":  GLOBAL_CATS,
    "marker_genes":       markers_common,
}
cfg_path = output_dir / "config_v3_0.json"
with open(cfg_path, 'w') as f:
    json.dump(viz_config, f, indent=2)
print(f"  [OK] {cfg_path}")

STEP 12: SAVE Merged DATA + CONFIG
  [sanitize] barcode: object(str) -> plain str
  [sanitize] schpl_pred_raw: object(str) -> plain str


  [sanitize] schpl_pred: object(str) -> plain str
  [sanitize] schpl_reject_type: object(str) -> plain str
  [sanitize] schpl_novel_candidate: object(bool+NA) -> bool
  [OK] /home/h2048/data/py/0330/bcell_merge_schpl_v3_0/reference_plus_query_merged_v3_0.h5ad  (0.05 GB)
  [OK] /home/h2048/data/py/0330/bcell_merge_schpl_v3_0/config_v3_0.json


## Section 16: Visualization — Merged Overview (6-panel)

In [45]:
print("=" * 80)
print("STEP 13: VIZ — MERGED OVERVIEW (6-panel)")
print("=" * 80)

marker_genes_list = adata_merged.uns.get(f"{MARKER_OBSM_KEY}_genes", []) if MARKER_EXPR_COMPARABLE else []
umap_coords       = adata_merged.obsm["X_umap"]

fig, axes = plt.subplots(2, 3, figsize=(24, 16))

sc.pl.umap(adata_merged, color=OBSKEY_SOURCE,
           ax=axes[0,0], show=False, title="Data Source",
           palette=PALETTE_SOURCE, frameon=False, s=20)

sc.pl.umap(adata_merged, color=VIZ_REF_ONLY,
           ax=axes[0,1], show=False, title="Reference Labels",
           legend_loc="right margin", palette=CT_PALETTE,
           frameon=False, s=20, na_color='lightgray')

sc.pl.umap(adata_merged, color=VIZ_QRY_FINAL,
           ax=axes[0,2], show=False, title="scArches Final Label (Query)",
           legend_loc="right margin", palette=CT_PALETTE,
           frameon=False, s=20, na_color='lightgray')

sc.pl.umap(adata_merged, color=VIZ_SCHPL_PRED,
           ax=axes[1,0], show=False, title="scHPL Prediction (Rejected=grey)",
           legend_loc="right margin", palette=CT_PALETTE_EXT,
           frameon=False, s=20, na_color='lightgray')

sc.pl.umap(adata_merged, color=VIZ_QRY_CONF,
           ax=axes[1,1], show=False, title="scArches Confidence (Query)",
           cmap=CMAP_CONFIDENCE, vmin=0, vmax=1,
           frameon=False, s=20, na_color='lightgray')

if MARKER_EXPR_COMPARABLE and "CD19" in marker_genes_list:
    idx  = marker_genes_list.index("CD19")
    expr = adata_merged.obsm[MARKER_OBSM_KEY][:, idx]
    sc_obj = axes[1,2].scatter(umap_coords[:,0], umap_coords[:,1],
                               c=expr, cmap=CMAP_EXPRESSION, s=20, rasterized=True)
    axes[1,2].set_title("CD19 Expression (merged comparable scale)", fontsize=14)
    axes[1,2].axis('off')
    plt.colorbar(sc_obj, ax=axes[1,2], fraction=0.046, pad=0.04)
else:
    axes[1,2].text(
        0.5, 0.5,
        "Merged marker panel disabled\n"
        f"ref={MARKER_SOURCE_REF}\n"
        f"query={MARKER_SOURCE_QRY}",
        ha='center', va='center', transform=axes[1,2].transAxes
    )
    axes[1,2].set_title("Marker panel omitted", fontsize=14)
    axes[1,2].axis('off')

plt.suptitle("B Cell Merge + scHPL v3.0", fontsize=13, fontweight='bold')
plt.tight_layout()
out = figures_dir / f"merged_overview_6panel_v3_0.{FIGURE_FORMAT}"
save_rasterized_figure(fig, out, dpi=DPI)
print(f"  [OK] {out.name}")

# Restore sc after matplotlib scatter
import scanpy as sc

STEP 13: VIZ — MERGED OVERVIEW (6-panel)


  [OK] merged_overview_6panel_v3_0.pdf


## Section 17: Visualization — Side-by-Side + Marker Panel

In [46]:
print("STEP 14a: VIZ — REFERENCE vs QUERY SIDE-BY-SIDE")

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
sc.pl.umap(adata_merged, color=VIZ_REF_ONLY,
           ax=axes[0], show=False, title="Reference: Original Labels",
           legend_loc="right margin", palette=CT_PALETTE,
           frameon=False, s=25, na_color='lightgray')
sc.pl.umap(adata_merged, color=VIZ_QRY_FINAL,
           ax=axes[1], show=False, title="Query: scArches Predicted Labels",
           legend_loc="right margin", palette=CT_PALETTE,
           frameon=False, s=25, na_color='lightgray')
plt.tight_layout()
out = figures_dir / f"reference_vs_query_sidebyside_v3_0.{FIGURE_FORMAT}"
save_rasterized_figure(fig, out, dpi=DPI)
print(f"  [OK] {out.name}")


print("\nSTEP 14b: VIZ — MARKER GENE PANEL")

if MARKER_EXPR_COMPARABLE:
    marker_genes_list = adata_merged.uns.get(f"{MARKER_OBSM_KEY}_genes", [])
    umap_coords       = adata_merged.obsm["X_umap"]
    marker_matrix     = adata_merged.obsm[MARKER_OBSM_KEY]
    panel_title       = "Merged comparable marker panel"
    out_name          = f"bcell_markers_panel_v3_0.{FIGURE_FORMAT}"
else:
    marker_genes_list = adata_query.uns.get(f"{MARKER_OBSM_KEY}_genes", [])
    umap_coords       = adata_query.obsm["X_umap"]
    marker_matrix     = adata_query.obsm[MARKER_OBSM_KEY]
    panel_title       = f"Query-only marker panel (source={MARKER_SOURCE_QRY}; not comparable to ref)"
    out_name          = f"bcell_markers_query_only_v3_0.{FIGURE_FORMAT}"
    print(f"  [INFO] Using query-only marker panel because ref/query sources differ: ref={MARKER_SOURCE_REF}, query={MARKER_SOURCE_QRY}")

if marker_genes_list:
    n_m   = len(marker_genes_list)
    ncols = 4
    nrows = int(np.ceil(n_m / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(20, 5*nrows))
    axes_flat = axes.flatten() if nrows > 1 else list(axes)
    for i, gene in enumerate(marker_genes_list):
        expr = marker_matrix[:, i]
        sc_obj = axes_flat[i].scatter(
            umap_coords[:,0], umap_coords[:,1],
            c=expr, cmap=CMAP_EXPRESSION, s=30, rasterized=True)
        axes_flat[i].set_title(f"{gene}", fontsize=12, fontweight='bold')
        axes_flat[i].axis('off')
        plt.colorbar(sc_obj, ax=axes_flat[i], fraction=0.046, pad=0.04)
    for j in range(i+1, len(axes_flat)):
        axes_flat[j].axis('off')
    plt.suptitle(panel_title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    out = figures_dir / out_name
    save_rasterized_figure(fig, out, dpi=DPI)
    print(f"  [OK] {out.name}")

import scanpy as sc

STEP 14a: VIZ — REFERENCE vs QUERY SIDE-BY-SIDE


  [OK] reference_vs_query_sidebyside_v3_0.pdf

STEP 14b: VIZ — MARKER GENE PANEL
  [INFO] Using query-only marker panel because ref/query sources differ: ref=layer:log1p, query=X
  [OK] bcell_markers_query_only_v3_0.pdf


## Section 18: Visualization — Query Detailed View + Confidence QC

In [23]:
print("STEP 15: VIZ — QUERY DETAILED VIEW")

qry_cells           = adata_merged.obs[OBSKEY_SOURCE] == "query"
adata_query_subset  = adata_merged[qry_cells]   # view
remove_unused_categories(adata_query_subset, VIZ_QRY_PRED)
remove_unused_categories(adata_query_subset, VIZ_QRY_FINAL)

fig, axes = plt.subplots(2, 2, figsize=(18, 18))
sc.pl.umap(adata_query_subset, color=VIZ_QRY_PRED,
           ax=axes[0,0], show=False, title="All scArches Predictions",
           legend_loc="right margin", palette=CT_PALETTE, frameon=False, s=30)
sc.pl.umap(adata_query_subset, color=VIZ_QRY_FINAL,
           ax=axes[0,1], show=False, title="scArches Final (conf>=0.5)",
           legend_loc="right margin", palette=CT_PALETTE, frameon=False, s=30)
sc.pl.umap(adata_query_subset, color=VIZ_QRY_CONF,
           ax=axes[1,0], show=False, title="scArches Confidence",
           cmap=CMAP_CONFIDENCE, vmin=0, vmax=1, frameon=False, s=30)
conf_vals = adata_query_subset.obs[VIZ_QRY_CONF].values
conf_vals = conf_vals[np.isfinite(conf_vals)]
axes[1,1].hist(conf_vals, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[1,1].axvline(0.5, color='red', linestyle='--', linewidth=2, label='Threshold=0.5')
axes[1,1].set_xlabel('Mapping Confidence')
axes[1,1].set_ylabel('Cell Count')
axes[1,1].set_title('Confidence Distribution')
axes[1,1].legend()
axes[1,1].grid(alpha=0.3)
axes[1,1].spines['top'].set_visible(False)
axes[1,1].spines['right'].set_visible(False)

plt.tight_layout()
out = figures_dir / f"query_detailed_view_v3_0.{FIGURE_FORMAT}"
save_rasterized_figure(fig, out, dpi=DPI)
print(f"  [OK] {out.name}")
gc.collect()

STEP 15: VIZ — QUERY DETAILED VIEW
  [OK] query_detailed_view_v3_0.pdf


13183

## Section 19: Visualization — scHPL Overview (new in v3.0)

In [47]:
print("STEP 16: VIZ — scHPL OVERVIEW")

fig, axes = plt.subplots(2, 3, figsize=(24, 16))

sc.pl.umap(adata_merged, color=VIZ_QRY_FINAL,
           ax=axes[0,0], show=False, title="scArches Final Label",
           legend_loc="right margin", palette=CT_PALETTE,
           frameon=False, s=20, na_color='lightgray')

sc.pl.umap(adata_merged, color=VIZ_SCHPL_PRED,
           ax=axes[0,1], show=False, title="scHPL Prediction (Rejected=grey)",
           legend_loc="right margin", palette=CT_PALETTE_EXT,
           frameon=False, s=20, na_color='lightgray')

sc.pl.umap(adata_merged, color=COL_SCHPL_REJECTED,
           ax=axes[0,2], show=False, title="scHPL Rejected Cells",
           frameon=False, s=20)

sc.pl.umap(adata_merged, color=COL_SCHPL_REJ_TYPE,
           ax=axes[1,0], show=False, title="Rejection Type (dist/RE/prob)",
           legend_loc="right margin", legend_fontsize=8,
           frameon=False, s=20)

sc.pl.umap(adata_merged, color=COL_SCHPL_PROB,
           ax=axes[1,1], show=False, title="scHPL Posterior Probability",
           cmap="viridis", vmin=0, vmax=1, frameon=False, s=20)

sc.pl.umap(adata_merged, color="schpl_novel_candidate",
           ax=axes[1,2], show=False, title="Novel Candidate Cells",
           frameon=False, s=20)

plt.suptitle("scHPL treeArches — B Cell v3.0", fontsize=13, fontweight='bold')
plt.tight_layout()
out = figures_dir / f"schpl_overview_6panel_v3_0.{FIGURE_FORMAT}"
save_rasterized_figure(fig, out, dpi=DPI)
print(f"  [OK] {out.name}")

STEP 16: VIZ — scHPL OVERVIEW


  [OK] schpl_overview_6panel_v3_0.pdf


## Section 20: Visualization — scArches vs scHPL Comparison (new in v3.0)

In [48]:
print("STEP 17: VIZ — scArches vs scHPL COMPARISON")

# Panel A: Side-by-side UMAP
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
sc.pl.umap(adata_merged, color=VIZ_QRY_FINAL,
           ax=axes[0], show=False, title="scArches Final Label (Query)",
           legend_loc="right margin", palette=CT_PALETTE,
           frameon=False, s=25, na_color='lightgray')
sc.pl.umap(adata_merged, color=VIZ_SCHPL_PRED,
           ax=axes[1], show=False, title="scHPL Prediction (Query; Rejected in grey)",
           legend_loc="right margin", palette=CT_PALETTE_EXT,
           frameon=False, s=25, na_color='lightgray')
plt.tight_layout()
out = figures_dir / f"schpl_vs_scarches_sidebyside_v3_0.{FIGURE_FORMAT}"
save_rasterized_figure(fig, out, dpi=DPI)
print(f"  [OK] {out.name}")


# Panel B: Rejection confidence analysis
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# [0] scHPL probability: accepted vs rejected
qry_m = adata_merged.obs[OBSKEY_SOURCE] == "query"
prob_all = adata_merged.obs.loc[qry_m, COL_SCHPL_PROB]
rej_flag = adata_merged.obs.loc[qry_m, COL_SCHPL_REJECTED]
prob_acc = prob_all[~rej_flag].dropna()
prob_rej = prob_all[rej_flag].dropna()
axes[0].hist(prob_acc, bins=50, alpha=0.7, color='steelblue', label='Accepted', density=True)
axes[0].hist(prob_rej, bins=50, alpha=0.7, color='tomato',    label='Rejected', density=True)
axes[0].axvline(SCHPL_REJ_THRESHOLD, color='red', linestyle='--',
               label=f'rej_threshold={SCHPL_REJ_THRESHOLD}')
axes[0].set_xlabel('scHPL Posterior Probability')
axes[0].set_ylabel('Density')
axes[0].set_title('Probability: Accepted vs Rejected')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# [1] Rejection type pie
rej_type_counts = adata_merged.obs.loc[qry_m, COL_SCHPL_REJ_TYPE].value_counts()
colors_pie = ['steelblue' if t == 'accepted' else 'tomato' for t in rej_type_counts.index]
axes[1].pie(rej_type_counts.values, labels=rej_type_counts.index,
            autopct='%1.1f%%', colors=colors_pie, startangle=90)
axes[1].set_title('Rejection Type Distribution')

# [2] scArches confidence: scHPL accepted vs rejected
conf_acc = adata_merged.obs.loc[qry_m & ~adata_merged.obs[COL_SCHPL_REJECTED], OBSKEY_CONF]
conf_rej = adata_merged.obs.loc[qry_m & adata_merged.obs[COL_SCHPL_REJECTED],  OBSKEY_CONF]
axes[2].hist(pd.to_numeric(conf_acc, errors='coerce').dropna(),
             bins=50, alpha=0.7, color='steelblue', label='scHPL Accepted', density=True)
axes[2].hist(pd.to_numeric(conf_rej, errors='coerce').dropna(),
             bins=50, alpha=0.7, color='tomato',    label='scHPL Rejected', density=True)
axes[2].axvline(0.5, color='black', linestyle='--', label='scArches threshold=0.5')
axes[2].set_xlabel('scArches Confidence')
axes[2].set_ylabel('Density')
axes[2].set_title('scArches Confidence for scHPL Accepted/Rejected')
axes[2].legend()
axes[2].grid(alpha=0.3)
axes[2].spines['top'].set_visible(False)
axes[2].spines['right'].set_visible(False)

plt.tight_layout()
out = figures_dir / f"schpl_rejection_analysis_v3_0.{FIGURE_FORMAT}"
save_rasterized_figure(fig, out, dpi=DPI)
print(f"  [OK] {out.name}")


# Panel C: Agreement heatmap (accepted cells, raw scArches prediction)
df_heat = adata_merged.obs.loc[
    qry_m & ~adata_merged.obs[COL_SCHPL_REJECTED],
    [OBSKEY_QRY_PRED, COL_SCHPL_PRED]
].copy()
df_heat.columns = ["scArches_pred", "scHPL_pred"]
ct = pd.crosstab(df_heat["scArches_pred"], df_heat["scHPL_pred"], normalize="index")

fig, ax = plt.subplots(figsize=(max(8, ct.shape[1]*0.9), max(6, ct.shape[0]*0.6)))
sns.heatmap(ct, ax=ax, cmap="Blues",
            annot=ct.shape[0] <= 20, fmt=".2f", linewidths=0.5,
            cbar_kws={"label": "Fraction of scArches cells"})
ax.set_xlabel("scHPL Prediction")
ax.set_ylabel("scArches Prediction")
ax.set_title("scArches pred vs scHPL Agreement (accepted cells, row-normalized)")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
out = figures_dir / f"schpl_vs_scarches_heatmap_v3_0.{FIGURE_FORMAT}"
save_rasterized_figure(fig, out, dpi=DPI)
print(f"  [OK] {out.name}")

STEP 17: VIZ — scArches vs scHPL COMPARISON


  [OK] schpl_vs_scarches_sidebyside_v3_0.pdf
  [OK] schpl_rejection_analysis_v3_0.pdf
  [OK] schpl_vs_scarches_heatmap_v3_0.pdf


## Section 21: Visualization — Novel Candidates (new in v3.0, conditional)

In [26]:
n_novel = adata_merged.obs["schpl_novel_candidate"].sum()

if n_novel > 0:
    print(f"STEP 18: VIZ — NOVEL CANDIDATES ({n_novel:,} cells)")
    fig, axes = plt.subplots(1, 3, figsize=(21, 7))
    sc.pl.umap(adata_merged, color="schpl_novel_candidate",
               ax=axes[0], show=False, frameon=False, s=4,
               title=f"Novel Candidates (n={n_novel:,})")
    sc.pl.umap(adata_merged, color="leiden_schpl_qc",
               ax=axes[1], show=False, frameon=False, s=4,
               legend_loc="right margin", legend_fontsize=7,
               title="Leiden Clusters (res=0.5)")
    sc.pl.umap(adata_merged, color=OBSKEY_CONF,
               ax=axes[2], show=False, frameon=False, s=4,
               cmap="RdYlGn", vmin=0, vmax=1,
               title="scArches Confidence")
    plt.tight_layout()
    out = figures_dir / f"schpl_novel_candidates_v3_0.{FIGURE_FORMAT}"
    save_rasterized_figure(fig, out, dpi=DPI)
    print(f"  [OK] {out.name}")
else:
    print("STEP 18: No novel candidate clusters detected — skipping panel.")

STEP 18: No novel candidate clusters detected — skipping panel.


## Section 22: QC Summary Table

In [49]:
print("STEP 19: QC SUMMARY TABLE")

qry_m = adata_merged.obs[OBSKEY_SOURCE] == "query"
df_qc = pd.DataFrame({
    'cell_type':  adata_merged.obs.loc[qry_m, VIZ_QRY_FINAL].astype("string"),
    'confidence': pd.to_numeric(adata_merged.obs.loc[qry_m, VIZ_QRY_CONF],
                                errors='coerce'),
    'schpl_pred': adata_merged.obs.loc[qry_m, COL_SCHPL_PRED].astype(str),
    'schpl_rej':  adata_merged.obs.loc[qry_m, COL_SCHPL_REJECTED],
}).dropna(subset=['cell_type', 'confidence'])

summary = df_qc.groupby('cell_type')['confidence'].agg(
    Count='count', Mean='mean', Median='median', Std='std', Min='min', Max='max'
).round(3)
summary['Pct']       = (summary['Count'] / summary['Count'].sum() * 100).round(2)
summary['Rej_n']     = df_qc.groupby('cell_type')['schpl_rej'].sum().astype(int)
summary['Rej_rate']  = (summary['Rej_n'] / summary['Count'] * 100).round(1)
summary = summary.sort_values('Count', ascending=False)

print(summary.to_string())

csv_path = output_dir / "qc_summary_by_celltype_v3_0.csv"
summary.to_csv(csv_path)
print(f"\n  [OK] {csv_path}")

# Violin plot
top_cts = summary.head(10).index
df_top  = df_qc[df_qc['cell_type'].isin(top_cts)]
ct_order = df_top.groupby('cell_type')['confidence'].median().sort_values(ascending=False).index

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.violinplot(data=df_top, x='cell_type', y='confidence', order=ct_order,
               ax=axes[0], palette='Set2')
axes[0].axhline(0.5, color='red', linestyle='--', linewidth=2, label='Threshold=0.5')
axes[0].set_title('scArches Confidence by Cell Type (Violin)')
axes[0].set_xlabel('Cell Type')
axes[0].set_ylabel('Mapping Confidence')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

sns.boxplot(data=df_top, x='cell_type', y='confidence', order=ct_order,
            ax=axes[1], palette='Set2')
axes[1].axhline(0.5, color='red', linestyle='--', linewidth=2, label='Threshold=0.5')
axes[1].set_title('scArches Confidence by Cell Type (Box)')
axes[1].set_xlabel('Cell Type')
axes[1].set_ylabel('Mapping Confidence')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
out = figures_dir / f"confidence_by_celltype_v3_0.{FIGURE_FORMAT}"
plt.savefig(out, dpi=DPI, bbox_inches='tight')
plt.close()
print(f"  [OK] {out.name}")

STEP 19: QC SUMMARY TABLE
                   Count   Mean  Median    Std    Min  Max    Pct  Rej_n  Rej_rate
cell_type                                                                         
Plasma             19902  0.998   1.000  0.024  0.501  1.0  48.29    213       1.1
Naive_B            11272  0.957   1.000  0.100  0.500  1.0  27.35    578       5.1
Memory_B            9126  0.944   0.998  0.113  0.500  1.0  22.14    141       1.5
Atypical_Memory_B    630  0.860   0.932  0.155  0.503  1.0   1.53     70      11.1
GC_B                 155  0.914   0.997  0.145  0.511  1.0   0.38      9       5.8
Unknown              132  0.453   0.463  0.042  0.313  0.5   0.32     25      18.9

  [OK] /home/h2048/data/py/0330/bcell_merge_schpl_v3_0/qc_summary_by_celltype_v3_0.csv
  [OK] confidence_by_celltype_v3_0.pdf


## Section 23: Save Label Comparison CSV + Final Summary

In [50]:
elapsed = (time.time() - PIPELINE_START) / 60

# Label comparison CSV
label_csv = output_dir / "label_comparison_v3_0.csv"
adata_merged.obs[[
    OBSKEY_SOURCE, OBSKEY_BATCH,
    OBSKEY_REF_LABEL, OBSKEY_QRY_PRED, OBSKEY_QRY_FINAL, OBSKEY_CONF,
    COL_SCHPL_RAW, COL_SCHPL_PRED, COL_SCHPL_PROB,
    COL_SCHPL_REJECTED, COL_SCHPL_REJ_TYPE,
    "leiden_schpl_qc", "schpl_novel_candidate",
]].to_csv(label_csv)
print(f"  [OK] {label_csv}")

# Final summary
print("\n" + "=" * 80)
print("FINAL SUMMARY — B Cell Merge + scHPL v3.0")
print("=" * 80)
print(f"Runtime     : {elapsed:.1f} min")
print(f"Merged shape: {adata_merged.shape}")
ref_n = (adata_merged.obs[OBSKEY_SOURCE] == "reference").sum()
qry_n = (adata_merged.obs[OBSKEY_SOURCE] == "query").sum()
print(f"  Reference : {ref_n:,}  Query: {qry_n:,}")
print(f"  Query UMAP basis : {adata_merged.uns.get('query_umap_basis', 'X_umap')}")
print(f"  Marker comparability : {MARKER_EXPR_COMPARABLE} (ref={MARKER_SOURCE_REF}, query={MARKER_SOURCE_QRY})")
print(f"\nscHPL")
print(f"  Accepted  : {n_accepted:,} ({n_accepted/n_total*100:.1f}%)")
print(f"  Rejected  : {n_rejected:,} ({rej_rate:.1f}%)")
print(f"    dist    : {int(mask_rej_dist.sum()):,}")
print(f"    RE      : {int(mask_rej_re.sum()):,}")
print(f"    prob    : {int(mask_rej_prob.sum()):,}")
print(f"  Novel cand: {adata_merged.obs['schpl_novel_candidate'].sum():,} "
      f"(clusters: {novel_clusters})")
print(f"  Agreement (scArches pred  vs scHPL, accepted): {agree_rate_pred:.1f}%")
print(f"  Agreement (scArches final vs scHPL, accepted): {agree_rate_final:.1f}%")
print(f"\nNew obs columns (query only; v2.0 cols unchanged):")
for col in [COL_SCHPL_RAW, COL_SCHPL_PRED, COL_SCHPL_PROB,
            COL_SCHPL_REJECTED, COL_SCHPL_REJ_TYPE,
            "leiden_schpl_qc", "schpl_novel_candidate"]:
    print(f"  {col}")
print(f"\nOutputs:")
print(f"  {merged_output}")
print(f"  {tree_path}")
print(f"  {label_csv}")
print(f"  {cfg_path}")
print(f"  {figures_dir}/  ({len(list(figures_dir.glob('*')))} files)")
print("\n" + "=" * 80)
print("SUCCESS")
print("=" * 80)

  [OK] /home/h2048/data/py/0330/bcell_merge_schpl_v3_0/label_comparison_v3_0.csv

FINAL SUMMARY — B Cell Merge + scHPL v3.0
Runtime     : 15.6 min
Merged shape: (52787, 3955)
  Reference : 11,570  Query: 41,217
  Query UMAP basis : X_umap_mapped
  Marker comparability : False (ref=layer:log1p, query=X)

scHPL
  Accepted  : 40,181 (97.5%)
  Rejected  : 1,036 (2.5%)
    dist    : 174
    RE      : 72
    prob    : 790
  Novel cand: 0 (clusters: [])
  Agreement (scArches pred  vs scHPL, accepted): 92.6%
  Agreement (scArches final vs scHPL, accepted): 92.4%

New obs columns (query only; v2.0 cols unchanged):
  schpl_pred_raw
  schpl_pred
  schpl_prob
  schpl_rejected
  schpl_reject_type
  leiden_schpl_qc
  schpl_novel_candidate

Outputs:
  /home/h2048/data/py/0330/bcell_merge_schpl_v3_0/reference_plus_query_merged_v3_0.h5ad
  /home/h2048/data/py/0330/bcell_merge_schpl_v3_0/schpl_tree_bcell_L2_v3_0.pkl
  /home/h2048/data/py/0330/bcell_merge_schpl_v3_0/label_comparison_v3_0.csv
  /home/h204